In [1]:
# Εισαγωγή των βιβλιοθηκών που χρειαζόμαστε
import pandas as pd
from pathlib import Path

# Ορισμός των διαδρομών των αρχείων
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "youtube_comments_raw.csv"
ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "annotations"
COMPLETED_ANNOTATIONS_DIR = ANNOTATIONS_DIR / "completed"

print("Raw dataset:", RAW_DATA_PATH)
print("Annotations folder:", ANNOTATIONS_DIR)
print("Completed annotations folder:", COMPLETED_ANNOTATIONS_DIR)

# Φόρτωση του raw dataset
df = pd.read_csv(RAW_DATA_PATH)

# Εμφάνιση βασικών πληροφοριών
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Raw dataset: /Users/loukito/Projects/TeamWork_Assignment/data/raw/youtube_comments_raw.csv
Annotations folder: /Users/loukito/Projects/TeamWork_Assignment/data/annotations
Completed annotations folder: /Users/loukito/Projects/TeamWork_Assignment/data/annotations/completed
Dataset loaded successfully!
Rows: 1000
Columns: 16


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,Ugyhj05JeMNyWipqxVp4AaABAg,@tokkyuuressha,UCu60UWqGRo_6NF6PDf-O1oQ,Football,football,8:34 what a clown piece from Ramos,2026-08-06T21:49:28Z,2026-08-06T21:49:28Z,0,0,moRX1XoBUFo,"Ronaldo, Messi, Neymar, Mbappe Shocked the Wor...",ArtSoccer,1,https://www.youtube.com/watch?v=moRX1XoBUFo&lc...,2026-08-07T09:46:01.721712+00:00
1,UgyhV_Isqm3AhYD0IOx4AaABAg,@따뜻한밥-t7q,UC7rljqa6-4WJPRuaeR9ho9Q,Football,football,저 ㅈㄹ을 해도 막판에 골을 못 넣는데...,2026-08-06T13:07:25Z,2026-08-06T13:07:25Z,0,0,02h7MB4Re4o,Best Football Skills 2026,SportsHD,2,https://www.youtube.com/watch?v=02h7MB4Re4o&lc...,2026-08-07T09:46:01.888698+00:00
2,UgxLBvsU6eZhWhcYA1x4AaABAg,@TheWingroveFamily,UCyY4jVInowKG9Nm20xXGHFw,Football,football,🚨YES GUYS! Who's team is better? Team Billy & ...,2026-07-10T15:08:13Z,2026-07-10T15:08:13Z,120,72,qHIiLIemo6k,INSANE WORLD CUP FC26 CARD BATTLE!!,The Wingrove Family,3,https://www.youtube.com/watch?v=qHIiLIemo6k&lc...,2026-08-07T09:46:02.075941+00:00
3,UgzYV_zK1zww1WW0Z5p4AaABAg,@Sathyabhama-z3d,UCkRfe1pHRCU3Weq3SJV40vQ,Football,football,Vbvgdff,2026-08-07T04:10:05Z,2026-08-07T04:10:05Z,0,1,pcAPOSv0OjA,SPAIN 1-0 ARGENTINA | FIFA WORLD CUP 2026 FINA...,BOBOLA TV,4,https://www.youtube.com/watch?v=pcAPOSv0OjA&lc...,2026-08-07T09:46:02.297134+00:00
4,Ugx-muWaVm7bSd_DNs94AaABAg,@AyomideOladipo-y2h,UCFVMGBH_VYmw_-LbpcLCvpw,Football,football,Volley looks easy but when you play football y...,2026-08-07T09:02:13Z,2026-08-07T09:02:13Z,0,0,ICQiBryLdMY,Most Inside Inside Foot Volleys In Football #f...,OG_Clips,5,https://www.youtube.com/watch?v=ICQiBryLdMY&lc...,2026-08-07T09:46:02.412103+00:00


In [2]:
# Βασικός έλεγχος του dataset πριν από το annotation
print("Total comments:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Missing comment IDs:", df["comment_id"].isna().sum())
print("Duplicate comment IDs:", df["comment_id"].duplicated().sum())
print("Empty comments:", df["text"].fillna("").str.strip().eq("").sum())

# Εμφάνιση του αριθμού σχολίων ανά topic
print("\nComments per topic:")
print(df["topic"].value_counts())

Total comments: 1000
Unique comment IDs: 1000
Missing comment IDs: 0
Duplicate comment IDs: 0
Empty comments: 0

Comments per topic:
topic
Football                   250
Climate Change             250
Video Games                250
Artificial Intelligence    250
Name: count, dtype: int64


In [3]:
# Επιλογή των στηλών που χρειάζονται για το annotation
annotation_df = df[["comment_id","author_name", "topic", "video_title", "video_channel", "text"]].copy()

# Δημιουργία κενής στήλης για τη χειροκίνητη αξιολόγηση
annotation_df["sentiment_label"] = ""

print("Annotation dataset created!")
print("Rows:", annotation_df.shape[0])
print("Columns:", annotation_df.shape[1])

display(annotation_df.head())

Annotation dataset created!
Rows: 1000
Columns: 7


,comment_id,author_name,topic,video_title,video_channel,text,sentiment_label
0,Ugyhj05JeMNyWipqxVp4AaABAg,@tokkyuuressha,Football,"Ronaldo, Messi, Neymar, Mbappe Shocked the Wor...",ArtSoccer,8:34 what a clown piece from Ramos,
1,UgyhV_Isqm3AhYD0IOx4AaABAg,@따뜻한밥-t7q,Football,Best Football Skills 2026,SportsHD,저 ㅈㄹ을 해도 막판에 골을 못 넣는데...,
2,UgxLBvsU6eZhWhcYA1x4AaABAg,@TheWingroveFamily,Football,INSANE WORLD CUP FC26 CARD BATTLE!!,The Wingrove Family,🚨YES GUYS! Who's team is better? Team Billy & ...,
3,UgzYV_zK1zww1WW0Z5p4AaABAg,@Sathyabhama-z3d,Football,SPAIN 1-0 ARGENTINA | FIFA WORLD CUP 2026 FINA...,BOBOLA TV,Vbvgdff,
4,Ugx-muWaVm7bSd_DNs94AaABAg,@AyomideOladipo-y2h,Football,Most Inside Inside Foot Volleys In Football #f...,OG_Clips,Volley looks easy but when you play football y...,


In [4]:
# Δημιουργία 3 ξεχωριστών αρχείων για κάθε annotator
# Και τα αποθηκεύουμε στον φάκελο annotations 
for annotator_number in range(1, 4):
    file_path = ANNOTATIONS_DIR / f"annotator_{annotator_number}.xlsx"
    
    annotation_df.to_excel(file_path, index=False)
    
    print(f"Created: {file_path.name}")

Created: annotator_1.xlsx
Created: annotator_2.xlsx
Created: annotator_3.xlsx


## Διαδικασία χειροκίνητου σχολιασμού

Σε αυτό το στάδιο, τα τρία αρχεία συμπληρώνονται χειροκίνητα και ανεξάρτητα από τους τρεις annotators(ομάδες):

* `annotator_1.xlsx`
* `annotator_2.xlsx`
* `annotator_3.xlsx`

Κάθε annotator αντιστοιχίζει σε κάθε σχόλιο μία ετικέτα συναισθήματος: `positive`, `neutral` ή `negative`. 

Μετά την ολοκλήρωση, τα συμπληρωμένα αρχεία αποθηκεύονται στον φάκελο `data/annotations/compl


In [5]:
# Διαδρομές των ολοκληρωμένων annotation αρχείων
ANNOTATOR_1_PATH = COMPLETED_ANNOTATIONS_DIR / "annotator_1_completed.xlsx"
ANNOTATOR_2_PATH = COMPLETED_ANNOTATIONS_DIR / "annotator_2_completed.xlsx"
ANNOTATOR_3_PATH = COMPLETED_ANNOTATIONS_DIR / "annotator_3_completed.xlsx"

# Φόρτωση των τριών αρχείων
annotator_1_df = pd.read_excel(ANNOTATOR_1_PATH)
annotator_2_df = pd.read_excel(ANNOTATOR_2_PATH)
annotator_3_df = pd.read_excel(ANNOTATOR_3_PATH)

print("Τα completed annotation αρχεία φορτώθηκαν επιτυχώς!")
print("Annotator 1:", annotator_1_df.shape)
print("Annotator 2:", annotator_2_df.shape)
print("Annotator 3:", annotator_3_df.shape)

Τα completed annotation αρχεία φορτώθηκαν επιτυχώς!
Annotator 1: (1000, 7)
Annotator 2: (1000, 7)
Annotator 3: (1000, 7)


In [6]:
# Επιτρεπόμενες ετικέτες συναισθήματος
VALID_LABELS = {"positive", "neutral", "negative"}

annotators = {
    "Annotator 1": annotator_1_df,
    "Annotator 2": annotator_2_df,
    "Annotator 3": annotator_3_df
}

# Έλεγχος κάθε αρχείου
for name, annotator_df in annotators.items():

    # Κανονικοποίηση των labels
    annotator_df["sentiment_label"] = (
        annotator_df["sentiment_label"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    labels = annotator_df["sentiment_label"]

    empty_labels = labels.isna() | labels.eq("")
    invalid_labels = ~labels.isin(VALID_LABELS) & ~empty_labels

    print(f"\n{name}")
    print("Συνολικές εγγραφές:", len(annotator_df))
    print("Κενά labels:", empty_labels.sum())
    print("Μη έγκυρα labels:", invalid_labels.sum())
    print("Διπλότυπα comment_id:", annotator_df["comment_id"].duplicated().sum())

# Έλεγχος ότι τα comment_id είναι ίδια και με την ίδια σειρά
same_comment_order = (
    annotator_1_df["comment_id"].equals(annotator_2_df["comment_id"])
    and annotator_1_df["comment_id"].equals(annotator_3_df["comment_id"])
)

print("\nΊδια comment_id και ίδια σειρά:", same_comment_order)


Annotator 1
Συνολικές εγγραφές: 1000
Κενά labels: 0
Μη έγκυρα labels: 0
Διπλότυπα comment_id: 0

Annotator 2
Συνολικές εγγραφές: 1000
Κενά labels: 0
Μη έγκυρα labels: 0
Διπλότυπα comment_id: 0

Annotator 3
Συνολικές εγγραφές: 1000
Κενά labels: 0
Μη έγκυρα labels: 0
Διπλότυπα comment_id: 0

Ίδια comment_id και ίδια σειρά: True


In [7]:
# Βασικές στήλες που θα διατηρηθούν
BASE_COLUMNS = [
    "comment_id",
    "author_name",
    "topic",
    "video_title",
    "video_channel",
    "text"
]

# Δημιουργία του ενιαίου annotation DataFrame
annotations_df = (
    annotator_1_df[BASE_COLUMNS + ["sentiment_label"]]
    .rename(columns={"sentiment_label": "annotator_1_label"})
    .merge(
        annotator_2_df[["comment_id", "sentiment_label"]].rename(
            columns={"sentiment_label": "annotator_2_label"}
        ),
        on="comment_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        annotator_3_df[["comment_id", "sentiment_label"]].rename(
            columns={"sentiment_label": "annotator_3_label"}
        ),
        on="comment_id",
        how="inner",
        validate="one_to_one"
    )
)

print("Διαστάσεις ενιαίου DataFrame:", annotations_df.shape)

display(
    annotations_df[
        [
            "comment_id",
            "annotator_1_label",
            "annotator_2_label",
            "annotator_3_label"
        ]
    ].head(10)
)

Διαστάσεις ενιαίου DataFrame: (1000, 9)


,comment_id,annotator_1_label,annotator_2_label,annotator_3_label
0,UgxrCmSeVh41IcaQYHV4AaABAg,positive,positive,positive
1,UgzYV_zK1zww1WW0Z5p4AaABAg,neutral,neutral,neutral
2,UgyR2PQD_ZodUbZ-4l54AaABAg,positive,positive,positive
3,UgzLvjrmCElWu91BsTN4AaABAg,neutral,positive,positive
4,UgwEwc_rrl-OKmA0J_54AaABAg,positive,positive,positive
5,Ugyhj05JeMNyWipqxVp4AaABAg,negative,negative,negative
6,UgzkYdjfSBCmg3p0JfB4AaABAg,negative,neutral,positive
7,UgxLBvsU6eZhWhcYA1x4AaABAg,neutral,neutral,neutral
8,UgyhV_Isqm3AhYD0IOx4AaABAg,negative,negative,negative
9,Ugwgs8WvNb146n7wMZV4AaABAg,positive,positive,positive


In [8]:
from statsmodels.stats.inter_rater import fleiss_kappa

# Στήλες με τις αξιολογήσεις των τριών annotators
ANNOTATOR_COLUMNS = [
    "annotator_1_label",
    "annotator_2_label",
    "annotator_3_label"
]

LABEL_ORDER = ["negative", "neutral", "positive"]

# Πλήθος διαφορετικών labels ανά σχόλιο
unique_labels_per_comment = annotations_df[ANNOTATOR_COLUMNS].nunique(axis=1)

full_agreement = (unique_labels_per_comment == 1).sum()
majority_agreement = (unique_labels_per_comment == 2).sum()
complete_disagreement = (unique_labels_per_comment == 3).sum()

total_comments = len(annotations_df)

# Δημιουργία του πίνακα που απαιτείται για το Fleiss' Kappa
agreement_matrix = pd.DataFrame({
    label: annotations_df[ANNOTATOR_COLUMNS].eq(label).sum(axis=1)
    for label in LABEL_ORDER
})

# Κάθε γραμμή πρέπει να έχει συνολικά 3 αξιολογήσεις
assert agreement_matrix.sum(axis=1).eq(3).all()

# Υπολογισμός Fleiss' Kappa
kappa_score = fleiss_kappa(
    agreement_matrix.to_numpy(),
    method="fleiss"
)

print(
    f"Πλήρης Ομοφωνία: {full_agreement} "
    f"({full_agreement / total_comments:.1%})"
)

print(
    f"Συμφωνία πλειοψηφίας: {majority_agreement} "
    f"({majority_agreement / total_comments:.1%})"
)

print(
    f"Πλήρης Διαφωνία: {complete_disagreement} "
    f"({complete_disagreement / total_comments:.1%})"
)

print(f"Fleiss' Kappa: {kappa_score:.3f}")

Πλήρης Ομοφωνία: 891 (89.1%)
Συμφωνία πλειοψηφίας: 104 (10.4%)
Πλήρης Διαφωνία: 5 (0.5%)
Fleiss' Kappa: 0.888


In [9]:
# Συνάρτηση για majority vote
def majority_vote(row):
    label_counts = row.value_counts()

    # Αποδοχή μόνο όταν τουλάχιστον 2 annotators συμφωνούν
    if label_counts.iloc[0] >= 2:
        return label_counts.index[0]

    # Και οι 3 annotators επέλεξαν διαφορετικό label
    return pd.NA


# Δημιουργία τελικού label όπου υπάρχει ομοφωνία ή πλειοψηφία
annotations_df["sentiment_label"] = annotations_df[
    ANNOTATOR_COLUMNS
].apply(majority_vote, axis=1)


# Εντοπισμός περιπτώσεων που χρειάζονται τελική απόφαση
adjudication_df = annotations_df[
    annotations_df["sentiment_label"].isna()
].copy()


print(
    "Labels που επιλύθηκαν:",
    annotations_df["sentiment_label"].notna().sum()
)

print(
    "Περιπτώσεις για αξιολογηση και αναθεώρηση:",
    len(adjudication_df)
)


display(
    adjudication_df[
        [
            "comment_id",
            "text",
            "annotator_1_label",
            "annotator_2_label",
            "annotator_3_label"
        ]
    ]
)

Labels που επιλύθηκαν: 995
Περιπτώσεις για αξιολογηση και αναθεώρηση: 5


,comment_id,text,annotator_1_label,annotator_2_label,annotator_3_label
6,UgzkYdjfSBCmg3p0JfB4AaABAg,"Trust me, even I can score that 1CM goal 😏",negative,neutral,positive
343,Ugxx5fqMiaK3kTQM-SR4AaABAg,😂😂😂,neutral,negative,positive
469,UgwGgbHCnNVnCrDdq9Z4AaABAg,"The Earth whispers softly, hear its call, \nA...",neutral,positive,negative
533,UgwbBxpZSjvRR6raatl4AaABAg,They put that in a fifa World Cup 😭😭😭,neutral,positive,negative
820,UgxpHS6BopzsZicc-x94AaABAg,Dude telling the history of computers lol..,negative,neutral,positive


In [10]:
# Τελική απόφαση για τις περιπτώσεις χωρίς πλειοψηφία
ADJUDICATED_LABELS = {
    "UgzkYdjfSBCmg3p0JfB4AaABAg": "negative",
    "Ugxx5fqMiaK3kTQM-SR4AaABAg": "positive",
    "UgwGgbHCnNVnCrDdq9Z4AaABAg": "positive",
    "UgwbBxpZSjvRR6raatl4AaABAg": "negative",
    "UgxpHS6BopzsZicc-x94AaABAg": "negative"
}

# Συμπλήρωση του τελικού sentiment_label μέσω comment_id
adjudication_mask = annotations_df["comment_id"].isin(
    ADJUDICATED_LABELS
)

annotations_df.loc[
    adjudication_mask,
    "sentiment_label"
] = (
    annotations_df.loc[adjudication_mask, "comment_id"]
    .map(ADJUDICATED_LABELS)
)

# Τελικός έλεγχος
remaining_missing = annotations_df["sentiment_label"].isna().sum()

print("Συνολικά τελικά labels:", annotations_df["sentiment_label"].notna().sum())
print("Labels που παραμένουν κενά:", remaining_missing)

print("\nΤελική κατανομή:")
print(annotations_df["sentiment_label"].value_counts())

assert remaining_missing == 0

Συνολικά τελικά labels: 1000
Labels που παραμένουν κενά: 0

Τελική κατανομή:
sentiment_label
negative    352
neutral     343
positive    305
Name: count, dtype: int64


In [11]:
# Αποθήκευση του τελικού annotated dataset
FINAL_DATA_DIR = PROJECT_ROOT / "data" / "final"
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

FINAL_COLUMNS = (
    BASE_COLUMNS
    + ANNOTATOR_COLUMNS
    + ["sentiment_label"]
)

final_annotated_df = annotations_df[FINAL_COLUMNS].copy()

# Τελικοί έλεγχοι ποιότητας
assert len(final_annotated_df) == 1000
assert final_annotated_df["comment_id"].notna().all()
assert final_annotated_df["comment_id"].is_unique
assert final_annotated_df["sentiment_label"].notna().all()
assert final_annotated_df["sentiment_label"].isin(VALID_LABELS).all()

# Αποθήκευση
FINAL_ANNOTATED_PATH = (
    FINAL_DATA_DIR / "youtube_comments_annotated.csv"
)

final_annotated_df.to_csv(
    FINAL_ANNOTATED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Το τελικό annotated dataset αποθηκεύτηκε!")
print("Αρχείο:", FINAL_ANNOTATED_PATH)
print("Διαστάσεις:", final_annotated_df.shape)
print(
    "Κενά τελικά labels:",
    final_annotated_df["sentiment_label"].isna().sum()
)

Το τελικό annotated dataset αποθηκεύτηκε!
Αρχείο: /Users/loukito/Projects/TeamWork_Assignment/data/final/youtube_comments_annotated.csv
Διαστάσεις: (1000, 10)
Κενά τελικά labels: 0
